In [ ]:
import h5py
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDOneClassSVM
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# --- 1. LOAD EMBEDDINGS FROM HDF5 ---
H5_PATH = "../embeddings/task1_virus_bacteria_Vir2vec-422M.h5"

with h5py.File(H5_PATH, "r") as f:
    X = np.array(f["embeddings"][:])
    raw_labels = [l.decode("utf-8") if isinstance(l, bytes) else str(l) for l in f["labels"][:]]

# Map Viral/Virus targets as Inliers (1) and Bacteria/Cellular as Anomalies (0)
y = np.array([1 if str(lbl).lower() in ["virus", "viral"] else 0 for lbl in raw_labels], dtype=np.int8)

# --- 2. ANOMALY ENSEMBLE FIT FUNCTION ---
def run_anomaly_ensemble(X_train, y_train, X_test, random_state=42):
    # Train strictly on viral inliers
    inlier_mask = (y_train == 1)
    Xv_train = X_train[inlier_mask]
    
    # Standardize & PCA
    scaler = StandardScaler().fit(Xv_train)
    Xv_tr_std = scaler.transform(Xv_train)
    X_tr_std, X_te_std = scaler.transform(X_train), scaler.transform(X_test)
    
    pca = PCA(n_components=min(512, X.shape[1]), random_state=random_state).fit(Xv_tr_std)
    Xv_tr_pca = pca.transform(Xv_tr_std)
    X_tr_pca, X_te_pca = pca.transform(X_tr_std), pca.transform(X_te_std)
    
    # 1. SGD-OCSVM + Nystroem Kernel
    gamma_scale = 1.0 / (Xv_tr_pca.shape[1] * np.var(Xv_tr_pca) + 1e-9)
    rbf = Nystroem(kernel="rbf", gamma=gamma_scale * 0.005, n_components=min(1000, len(Xv_tr_pca)), random_state=random_state)
    
    Xv_feat = rbf.fit_transform(Xv_tr_pca)
    X_tr_feat, X_te_feat = rbf.transform(X_tr_pca), rbf.transform(X_te_pca)
    
    oc = SGDOneClassSVM(nu=0.05, learning_rate="optimal", random_state=random_state).fit(Xv_feat)
    ocsvm_scores_tr = oc.decision_function(X_tr_feat)
    ocsvm_scores_te = oc.decision_function(X_te_feat)
    
    # 2. Isolation Forest
    iso = IsolationForest(n_estimators=200, contamination=0.05, bootstrap=True, random_state=random_state, n_jobs=-1).fit(Xv_tr_pca)
    iso_scores_tr = -iso.score_samples(X_tr_pca)
    iso_scores_te = -iso.score_samples(X_te_pca)
    
    # 3. Ensemble Weight & Quantile Threshold Calibration
    best_acc = -1.0
    best_w = (0.5, 0.5)
    best_th = 0.0
    
    for w1 in np.linspace(0.1, 0.9, 9):
        w2 = 1.0 - w1
        ens_tr = w1 * ocsvm_scores_tr + w2 * iso_scores_tr
        for q in np.linspace(0.01, 0.3, 15):
            th = np.quantile(ens_tr[y_train == 1], q)
            preds = (ens_tr > th).astype(int)
            acc = balanced_accuracy_score(y_train, preds)
            if acc > best_acc:
                best_acc = acc
                best_w = (w1, w2)
                best_th = th
                
    # Test Inference
    ens_te = best_w[0] * ocsvm_scores_te + best_w[1] * iso_scores_te
    ens_preds_te = (ens_te > best_th).astype(int)
    
    return ens_preds_te

# --- 3. 5-FOLD OUTER CROSS-VALIDATION ---
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    y_pred = run_anomaly_ensemble(X_train, y_train, X_test, random_state=42 + fold_idx)
    
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    fold_metrics.append({
        "fold": fold_idx,
        "balanced_accuracy": bal_acc,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0)
    })
    print(f"Fold {fold_idx}: Anomaly Ensemble Balanced Acc = {bal_acc:.4f}")

# --- 4. SUMMARY ---
df_res = pd.DataFrame(fold_metrics)
print("\n--- Final Anomaly Detection Ensemble Results ---")
print(f"Macro Balanced Accuracy: {df_res['balanced_accuracy'].mean():.4f} ± {df_res['balanced_accuracy'].std():.4f}")